Compared to TF-IDF, Sentence Transformers have better semantic matching, making it better for LLM prompts.

- TF-IDF: “truck accident Nairobi” ≠ “commercial vehicle crash Nairobi”
- Sentence-BERT turns full sentences into meaning-rich embeddings, enabling the system to match behavioral similarity and risk patterns, not just keywords.
- Retrieved policies become more contextually similar cases, not just word matches

BERT embeddings are however slower in speed and more compute-heavy.

In [2]:
import pandas as pd

df = pd.read_csv("insurance_dataset.csv")

In [3]:
import joblib

model = joblib.load("xgb_model.pkl")
feature_columns = joblib.load("feature_columns.pkl")

In [4]:
# Adding core features to the knowledge base
kb_features = ["age", "driver_type", "years_experience", "vehicle_type", "vehicle_value", "vehicle_age", "annual_mileage", "previous_claims", "risk_score", "airbags", "tracking_device", "region", "policy_duration"]

# Creating text representation of each policy
def policy_to_text(row):
    return (
        f"Policyholder: age {row['age']}, driver_type {row['driver_type']}, years_experience {row['years_experience']}, "
        f"vehicle_type {row['vehicle_type']}, vehicle_value {row['vehicle_value']}, ({row['vehicle_age']} yrs old), "
        f"annual_mileage {row['annual_mileage']}, previous_claims {row['previous_claims']}, "
        f"safety: airbags {row['airbags']}, tracking_device {row['tracking_device']}, "
        f"region {row['region']}, policy_duration {row['policy_duration']} months, "
        f"risk_score {row['risk_score']:.2f}."
    )

df["policy_text"] = df.apply(policy_to_text, axis=1)

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

model_st = SentenceTransformer("all-MiniLM-L6-v2")

# Encoding all policies
policy_vectors = model_st.encode(
    df["policy_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True # makes cosine similarity faster, more stable and accurate
)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [6]:
policy_vectors.shape

(50000, 384)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_similar_policies(query_text, top_k=5):
    query_vec = model_st.encode([query_text], normalize_embeddings=True)
    sims = cosine_similarity(query_vec, policy_vectors).flatten()
    top_idx = sims.argsort()[::-1][:top_k]

    return df.iloc[top_idx][["policy_text", "risk_score", "claim_occurred"]], sims[top_idx]

In [8]:
query = "Private driver, 28 yrs, sedan, 5 yrs old, 18000 km/yr, 0 previous claims, Nairobi, 12 months policy"
similar_policies, scores = retrieve_similar_policies(query)
similar_policies

,policy_text,risk_score,claim_occurred
12913,"Policyholder: age 26, driver_type private, yea...",0.55,0
36884,"Policyholder: age 24, driver_type private, yea...",0.55,0
8025,"Policyholder: age 56, driver_type private, yea...",0.53,0
12447,"Policyholder: age 26, driver_type private, yea...",0.73,0
12657,"Policyholder: age 60, driver_type private, yea...",0.35,0


In [9]:
def calculate_premium(predicted_risk, vehicle_value,
                      expense_loading=0.3, profit_margin=0.1):
    
    expected_loss = predicted_risk * vehicle_value
    
    premium = expected_loss * (1 + expense_loading + profit_margin)
    premium = max(premium, 5000)
    
    return expected_loss, premium

In [10]:
def predict_risk_from_query(query_dict):
    input_df = pd.DataFrame([query_dict])
    input_df = pd.get_dummies(input_df)
    input_df = input_df.reindex(columns=feature_columns, fill_value=0)
    
    return model.predict_proba(input_df)[:, 1][0]

In [11]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = ChatGroq(
    temperature=0.3,
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

def ask_groq(prompt):
    response = client.invoke([
        {"role": "system", "content": "You are an expert insurance underwriter."},
        {"role": "user", "content": prompt}
    ])
    
    return response.content

In [12]:
def build_prompt(user_question, retrieved_policies, predicted_risk, premium):
    
    context = "\n".join(retrieved_policies["policy_text"].tolist())
    
    prompt = f"""
You are an AI insurance underwriting assistant.

PREDICTED RISK SCORE: {predicted_risk:.2f}
RECOMMENDED PREMIUM: {premium:.0f}

SIMILAR POLICIES:
{context}

QUESTION:
{user_question}

INSTRUCTIONS:
- Provide underwriting decision (Approve / Conditional / Decline)
- Explain reasoning using risk score and similar cases
- Justify the recommended premium
- Highlight key risk drivers

ANSWER:
"""
    return prompt

In [13]:
query_features = {
    "age": 30,
    "driver_type": "private",
    "vehicle_age": 3,
    "vehicle_type": "sedan",
    "vehicle_value": 1200000,
    "annual_mileage": 15000,
    "previous_claims": 0,
    "airbags": 4,
    "tracking_device": 1,
    "region": "Nairobi",
    "policy_duration": 12,
    "speeding_score": "medium"
}

In [16]:
import requests

user_question = "What is the risk profile for a 30-year-old private driver with a 3-year-old sedan, valued at 1200000, annual mileage 15000, 4 airbags, 1 tracking device, in Nairobi, no previous claims?"

retrieved_policies, sims = retrieve_similar_policies(user_question)
predicted_risk = predict_risk_from_query(query_features)
expected_loss, premium = calculate_premium(predicted_risk, query_features["vehicle_value"])
prompt = build_prompt(user_question, retrieved_policies, predicted_risk, premium)
answer = ask_groq(prompt)

print(answer)

**Underwriting Decision:** Approve

**Reasoning:**
The predicted risk score of 0.11 is relatively low, indicating a lower risk profile. Comparing this to similar policies, we can see that the risk scores range from 0.34 to 0.67. The proposed policyholder's profile is more similar to the first similar policy (age 21, risk score 0.38), but with a lower risk score. This suggests that the proposed policyholder is a relatively low-risk driver.

The key factors contributing to this low risk score are:

* Low age (30 years old) compared to some of the similar policies (e.g., 62 years old)
* Newer vehicle (3 years old) with a higher value (1200000), which may indicate a more responsible owner
* Presence of 4 airbags and 1 tracking device, which enhance safety features
* No previous claims, which suggests a good driving history
* Annual mileage (15000) is moderate, which is not excessively high

**Recommended Premium:**
The recommended premium of 183369 is relatively high compared to the simila

In [17]:
vehicle_value = query_features["vehicle_value"]

expected_loss, recommended_premium = calculate_premium(predicted_risk, vehicle_value)

print(f"Expected Loss: {expected_loss:.2f}")
print(f"Recommended Premium: {recommended_premium:.2f}")

Expected Loss: 130977.70
Recommended Premium: 183368.77


In [18]:
risk_avg = retrieved_policies["risk_score"].mean()
claim_rate = retrieved_policies["claim_occurred"].mean()

print(f"Average risk_score of similar policies: {risk_avg:.2f}")
print(f"Claim rate among similar policies: {claim_rate:.2%}")

Average risk_score of similar policies: 0.47
Claim rate among similar policies: 0.00%
